# Tahap 3 — Case Retrieval (v3 — Deep Clean)

**Input**: `data/processed/cases.json`

**Output**:
- `models/tfidf_vectorizer.pkl`, `models/svm_pasal.pkl`, `models/nb_pasal.pkl`
- `models/indobert_embeddings.npy`
- `data/eval/queries.json`, `data/eval/retrieval_metrics.csv`

### Perbaikan v3
- Preprocessing identik dengan notebook 02 (deep clean)
- IndoBERT chunked embedding (teks penuh, bukan truncate 512 token)
- CUDA otomatis
- TF-IDF tanpa max_features limit

In [ ]:
import json
import pickle
import re
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from transformers import AutoModel, AutoTokenizer
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

In [ ]:
DATA_PATH = Path("../data/processed/cases.json")
EVAL_DIR  = Path("../data/eval")
MODEL_DIR = Path("../models")

for d in [EVAL_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    cases = json.load(f)

print(f"Total kasus: {len(cases)}")
print(f"Distribusi: {dict(Counter([c['label_pasal'] for c in cases]))}")
print(f"Avg text_full: {np.mean([len(c['text_full']) for c in cases]):.0f} chars")

Device: cuda
Total kasus: 60
Distribusi: {'pasal_362': 30, 'pasal_363': 30}
Avg text_full: 43472 chars


## 3.1 Preprocessing

> `text_full` sudah di-deep-clean oleh notebook 02. Di sini kita hanya perlu preprocessing ringan.

In [ ]:
def preprocess(text):
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text


for c in cases:
    c["text_processed"] = preprocess(c["text_full"])

labels_pasal = [c["label_pasal"] for c in cases]
train_cases, test_cases = train_test_split(
    cases, test_size=0.2, stratify=labels_pasal, random_state=42
)

train_texts        = [c["text_processed"] for c in train_cases]
test_texts         = [c["text_processed"] for c in test_cases]
train_labels_pasal = [c["label_pasal"] for c in train_cases]
test_labels_pasal  = [c["label_pasal"] for c in test_cases]

print(f"Train: {len(train_cases)} | Test: {len(test_cases)}")
print(f"Train: {dict(Counter(train_labels_pasal))}")
print(f"Test : {dict(Counter(test_labels_pasal))}")

Train: 48 | Test: 12
Train: {'pasal_363': 24, 'pasal_362': 24}
Test : {'pasal_362': 6, 'pasal_363': 6}


In [ ]:
def evaluate_model(y_true, y_pred, model_name, label_name):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    print(f"\n{model_name} | {label_name}")
    print(f"  Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}")
    print(classification_report(y_true, y_pred, zero_division=0))
    return {"model": model_name, "label": label_name,
            "accuracy": round(acc, 4), "precision": round(prec, 4),
            "recall": round(rec, 4), "f1_score": round(f1, 4)}


def retrieval_predict(query_vectors, train_vectors, train_labels, k=5):
    predictions = []
    for i in range(query_vectors.shape[0]):
        sims = cosine_similarity(query_vectors[i:i+1], train_vectors).flatten()
        top_k_idx = sims.argsort()[::-1][:k]
        top_k_labels = [train_labels[j] for j in top_k_idx]
        predictions.append(Counter(top_k_labels).most_common(1)[0][0])
    return predictions


all_metrics = []

## 3.2 TF-IDF + ML

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=None, ngram_range=(1, 2),
    sublinear_tf=True, min_df=1, max_df=0.95,
)
X_train = vectorizer.fit_transform(train_texts)
X_test  = vectorizer.transform(test_texts)
all_tfidf_vectors = vectorizer.transform([c["text_processed"] for c in cases])

print(f"TF-IDF: train={X_train.shape}, test={X_test.shape}")
print(f"Vocab: {len(vectorizer.vocabulary_)}")

with open(MODEL_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
print("TF-IDF vectorizer disimpan")

TF-IDF: train=(48, 58392), test=(12, 58392)
Vocab: 58392
TF-IDF vectorizer disimpan


In [ ]:
svm_pasal = LinearSVC(max_iter=10000, random_state=42, C=1.0)
svm_pasal.fit(X_train, train_labels_pasal)
svm_pred = svm_pasal.predict(X_test)
all_metrics.append(evaluate_model(test_labels_pasal, svm_pred, "TF-IDF + SVM", "label_pasal"))

with open(MODEL_DIR / "svm_pasal.pkl", "wb") as f:
    pickle.dump(svm_pasal, f)


TF-IDF + SVM | label_pasal
  Acc=0.8333  Prec=0.8333  Rec=0.8333  F1=0.8333
              precision    recall  f1-score   support

   pasal_362       0.83      0.83      0.83         6
   pasal_363       0.83      0.83      0.83         6

    accuracy                           0.83        12
   macro avg       0.83      0.83      0.83        12
weighted avg       0.83      0.83      0.83        12



In [ ]:
nb_pasal = MultinomialNB(alpha=1.0)
nb_pasal.fit(X_train, train_labels_pasal)
nb_pred = nb_pasal.predict(X_test)
all_metrics.append(evaluate_model(test_labels_pasal, nb_pred, "TF-IDF + NaiveBayes", "label_pasal"))

with open(MODEL_DIR / "nb_pasal.pkl", "wb") as f:
    pickle.dump(nb_pasal, f)


TF-IDF + NaiveBayes | label_pasal
  Acc=0.7500  Prec=0.7571  Rec=0.7500  F1=0.7483
              precision    recall  f1-score   support

   pasal_362       0.80      0.67      0.73         6
   pasal_363       0.71      0.83      0.77         6

    accuracy                           0.75        12
   macro avg       0.76      0.75      0.75        12
weighted avg       0.76      0.75      0.75        12



In [ ]:
tfidf_ret_pred = retrieval_predict(X_test, X_train, train_labels_pasal, k=5)
all_metrics.append(evaluate_model(test_labels_pasal, tfidf_ret_pred, "TF-IDF + Cosine (k=5)", "label_pasal"))


TF-IDF + Cosine (k=5) | label_pasal
  Acc=0.7500  Prec=0.7571  Rec=0.7500  F1=0.7483
              precision    recall  f1-score   support

   pasal_362       0.80      0.67      0.73         6
   pasal_363       0.71      0.83      0.77         6

    accuracy                           0.75        12
   macro avg       0.76      0.75      0.75        12
weighted avg       0.76      0.75      0.75        12



## 3.3 IndoBERT Embedding (Chunked)

In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p1"
print(f"Loading {MODEL_NAME} → {device}")
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert_model.eval()
print("OK")

Loading indobenchmark/indobert-base-p1 → cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

OK


In [ ]:
def get_chunked_embedding(text, tokenizer, model, device, chunk_size=510):
    tokens = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    chunks = [tokens[i:i+chunk_size] for i in range(0, len(tokens), chunk_size)]

    chunk_embs = []
    for chunk in chunks:
        input_ids = torch.cat([
            torch.tensor([tokenizer.cls_token_id]),
            chunk,
            torch.tensor([tokenizer.sep_token_id])
        ]).unsqueeze(0).to(device)
        attention_mask = torch.ones_like(input_ids).to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        mask = attention_mask.unsqueeze(-1).float()
        emb = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
        chunk_embs.append(emb)

    if chunk_embs:
        return torch.stack(chunk_embs).mean(dim=0).cpu().numpy()[0]
    return np.zeros(model.config.hidden_size)

In [ ]:
all_texts = [c["text_processed"] for c in cases]

print(f"Generating chunked embeddings: {len(all_texts)} dokumen")
embeddings = []
for text in tqdm(all_texts, desc="IndoBERT embeddings"):
    emb = get_chunked_embedding(text, tokenizer, bert_model, device)
    embeddings.append(emb)

all_emb = np.array(embeddings)
np.save(MODEL_DIR / "indobert_embeddings.npy", all_emb)
print(f"Shape: {all_emb.shape}")

Generating chunked embeddings: 60 dokumen


IndoBERT embeddings:   0%|          | 0/60 [00:00<?, ?it/s]

Shape: (60, 768)


In [ ]:
case_ids  = [c["case_id"] for c in cases]
train_idx = [case_ids.index(c["case_id"]) for c in train_cases]
test_idx  = [case_ids.index(c["case_id"]) for c in test_cases]

train_emb = all_emb[train_idx]
test_emb  = all_emb[test_idx]

bert_ret_pred = retrieval_predict(test_emb, train_emb, train_labels_pasal, k=5)
all_metrics.append(evaluate_model(test_labels_pasal, bert_ret_pred, "IndoBERT + Cosine (k=5)", "label_pasal"))


IndoBERT + Cosine (k=5) | label_pasal
  Acc=0.5833  Prec=0.7727  Rec=0.5833  F1=0.4958
              precision    recall  f1-score   support

   pasal_362       1.00      0.17      0.29         6
   pasal_363       0.55      1.00      0.71         6

    accuracy                           0.58        12
   macro avg       0.77      0.58      0.50        12
weighted avg       0.77      0.58      0.50        12



## 3.4 Generate queries.json

In [ ]:
tfidf_case_ids = [c["case_id"] for c in cases]


def retrieve_tfidf(query_text, k=5):
    q_vec = vectorizer.transform([preprocess(query_text)])
    sims  = cosine_similarity(q_vec, all_tfidf_vectors).flatten()
    top_k = sims.argsort()[::-1][:k]
    return [{"case_id": tfidf_case_ids[i], "label": cases[i]["label_pasal"],
             "similarity": round(float(sims[i]), 4)} for i in top_k]


def retrieve_bert(query_text, k=5):
    q_emb = get_chunked_embedding(preprocess(query_text), tokenizer, bert_model, device).reshape(1, -1)
    sims  = cosine_similarity(q_emb, all_emb).flatten()
    top_k = sims.argsort()[::-1][:k]
    return [{"case_id": tfidf_case_ids[i], "label": cases[i]["label_pasal"],
             "similarity": round(float(sims[i]), 4)} for i in top_k]


queries = []
for i, tc in enumerate(test_cases):
    tfidf_results = retrieve_tfidf(tc["text_processed"], k=5)
    bert_results  = retrieve_bert(tc["text_processed"], k=5)
    queries.append({
        "query_id":               f"q{i+1:02d}",
        "source_case_id":         tc["case_id"],
        "ground_truth_label_pasal": tc["label_pasal"],
        "ground_truth_vonis_bulan": tc["vonis_bulan"],
        "query_text":             tc["text_processed"][:500],
        "tfidf_top5":             tfidf_results,
        "indobert_top5":          bert_results,
    })
    print(f"  q{i+1:02d}: {tc['case_id']} ({tc['label_pasal']}, {tc['vonis_bulan']} bln) "
          f"| TF-IDF={tfidf_results[0]['case_id']} | BERT={bert_results[0]['case_id']}")

with open(EVAL_DIR / "queries.json", "w", encoding="utf-8") as f:
    json.dump(queries, f, ensure_ascii=False, indent=2)
print(f"\nqueries.json: {len(queries)} queries")

  q01: case_pasal_362_011 (pasal_362, 12 bln) | TF-IDF=case_pasal_362_011 | BERT=case_pasal_362_011


  q02: case_pasal_363_029 (pasal_363, 10 bln) | TF-IDF=case_pasal_363_029 | BERT=case_pasal_363_029


  q03: case_pasal_363_015 (pasal_363, 0 bln) | TF-IDF=case_pasal_363_028 | BERT=case_pasal_363_028


  q04: case_pasal_362_007 (pasal_362, 5 bln) | TF-IDF=case_pasal_362_007 | BERT=case_pasal_362_007


  q05: case_pasal_363_016 (pasal_363, 0 bln) | TF-IDF=case_pasal_363_016 | BERT=case_pasal_363_016


...

  q08: case_pasal_362_008 (pasal_362, 10 bln) | TF-IDF=case_pasal_362_008 | BERT=case_pasal_362_008


  q09: case_pasal_363_026 (pasal_363, 0 bln) | TF-IDF=case_pasal_363_028 | BERT=case_pasal_363_028


  q10: case_pasal_362_020 (pasal_362, 12 bln) | TF-IDF=case_pasal_362_020 | BERT=case_pasal_362_020


  q11: case_pasal_363_028 (pasal_363, 0 bln) | TF-IDF=case_pasal_363_028 | BERT=case_pasal_363_028


  q12: case_pasal_362_021 (pasal_362, 24 bln) | TF-IDF=case_pasal_362_021 | BERT=case_pasal_362_021

queries.json: 12 queries


## 3.5 Retrieval Metrics Summary

In [ ]:
metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(EVAL_DIR / "retrieval_metrics.csv", index=False)
print("RETRIEVAL METRICS SUMMARY")
metrics_df

RETRIEVAL METRICS SUMMARY


,model,label,accuracy,precision,recall,f1_score
0,TF-IDF + SVM,label_pasal,0.8333,0.8333,0.8333,0.8333
1,TF-IDF + NaiveBayes,label_pasal,0.7500,0.7571,0.7500,0.7483
2,TF-IDF + Cosine (k=5),label_pasal,0.7500,0.7571,0.7500,0.7483
3,IndoBERT + Cosine (k=5),label_pasal,0.5833,0.7727,0.5833,0.4958
